# 05 - Risk scores, Prosecution Priority Score and baseline scenarios (SPEC sections 8.1-8.3)
* `risk_score`: P(collapse) at 12 and 24 months from the `champion` models, with a band (Low / Elevated / High / Critical), the top 3 drivers in plain English, and the model version.
* `scoring_weights`: versioned PPS criterion weights (defaults from the spec; ASIC must confirm them).
* `intervention_presets`: preset bundles of input changes, e.g. asset freeze first.
* SQL function `pps_compute(...)`: one transparent PPS formula shared by batch scoring, SQL what-ifs and the app.
* `pps_score`: baseline PPS per open case. `scenario`: a baseline plus one scenario per intervention for each case.

In [0]:
%pip install -q shap scikit-learn mlflow
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
CATALOG, SCHEMA = dbutils.widgets.get("catalog"), dbutils.widgets.get("schema")
assert CATALOG and SCHEMA
FQ = f"{CATALOG}.{SCHEMA}"
WEIGHTS_VERSION = "spec-default-v1"
FEATURES = ["company_age_years", "is_licensee", "ato_debt_latest_aud", "ato_debt_trend_6m_pct", "lodgement_on_time_pct",
            "stp_headcount_trend_pct", "gst_cancelled", "smr_count_24m", "large_flow_smr_count_24m", "apra_flag_count",
            "afca_complaints_latest", "afca_complaints_trend_pct", "breach_reports_12m", "enforcement_alternatives_count",
            "banned_director_count", "director_prior_failed_companies", "related_collapsed_entities"]
LABELS = {"company_age_years": "Company age", "is_licensee": "Holds an ASIC licence", "ato_debt_latest_aud": "ATO debt",
          "ato_debt_trend_6m_pct": "ATO debt trend (6 months)", "lodgement_on_time_pct": "On-time lodgement rate",
          "stp_headcount_trend_pct": "Payroll headcount trend", "gst_cancelled": "GST registration cancelled",
          "smr_count_24m": "AUSTRAC SMRs (24 months)", "large_flow_smr_count_24m": "Large-flow SMRs",
          "apra_flag_count": "APRA prudential flags", "afca_complaints_latest": "AFCA complaints",
          "afca_complaints_trend_pct": "AFCA complaint trend", "breach_reports_12m": "Breach reports (12 months)",
          "enforcement_alternatives_count": "Prior EUs or infringement notices", "banned_director_count": "Directors on the banned register",
          "director_prior_failed_companies": "Directors with prior failed companies", "related_collapsed_entities": "Related entities that collapsed"}

In [0]:
import json, mlflow, numpy as np, pandas as pd, shap
from pyspark.sql import functions as F, Window as W
mlflow.set_registry_uri("databricks-uc")
client = mlflow.MlflowClient()

latest = (spark.table(f"{FQ}.gold_entity_features")
          .withColumn("rn", F.row_number().over(W.partitionBy("entity_id").orderBy(F.desc("as_of_date")))).filter("rn = 1").drop("rn").toPandas())
X = latest[FEATURES].astype(float).fillna(0.0)
out = latest[["entity_id", "acn", "as_of_date"]].copy()
versions = {}
for h in (12, 24):
    mv = client.get_model_version_by_alias(f"{FQ}.collapse_model_{h}m", "champion")
    versions[h] = f"collapse_model_{h}m/v{mv.version}"
    model = mlflow.sklearn.load_model(f"models:/{FQ}.collapse_model_{h}m@champion")
    out[f"p_collapse_{h}m"] = model.predict_proba(X)[:, 1]
    if h == 12:
        sv = shap.Explainer(model.predict_proba, X.sample(min(200, len(X)), random_state=0))(X).values[..., 1]

def drivers(i):
    top = np.argsort(-np.abs(sv[i]))[:3]
    return json.dumps([{"feature": FEATURES[j], "label": LABELS[FEATURES[j]], "value": float(X.iloc[i, j]),
                        "direction": "raises" if sv[i, j] > 0 else "lowers", "shap": round(float(sv[i, j]), 4)} for j in top])

out["drivers_json"] = [drivers(i) for i in range(len(out))]
out["band"] = pd.cut(out.p_collapse_12m, [-0.01, 0.15, 0.35, 0.6, 1.0], labels=["Low", "Elevated", "High", "Critical"]).astype(str)
out["model_version"] = f"{versions[12]};{versions[24]}"
(spark.createDataFrame(out).withColumn("scored_at", F.current_timestamp()).withColumn("provenance", F.lit("derived"))
      .write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{FQ}.risk_score"))
display(spark.table(f"{FQ}.risk_score").groupBy("band").count())

In [0]:
spark.createDataFrame([(WEIGHTS_VERSION, 0.25, 0.20, 0.20, 0.20, 0.15, "SPEC 8.2 defaults - ASIC to confirm")],
    "weights_version string, w_financial double, w_success double, w_harm double, w_deterrence double, w_urgency double, note string") \
    .withColumn("created_at", F.current_timestamp()).write.mode("overwrite").saveAsTable(f"{FQ}.scoring_weights")

# Each preset is a bundle of multipliers / overrides applied to the case inputs
spark.createDataFrame([
    ("asset_freeze_first", "Seek asset freeze first", 1.10, 1.0, 0.20, None, 1.0),
    ("enforceable_undertaking", "Enforceable undertaking instead", 0.25, 0.6, None, 0.90, 0.5),
    ("litigation_cost_up_40", "Litigation cost +40%", 1.40, 1.0, None, None, 1.0),
    ("expedited_hearing", "Expedited hearing", 0.90, 0.6, None, None, 1.0)],
    "preset_id string, name string, cost_multiplier double, duration_multiplier double, p_collapse_override double, p_success_override double, recoverable_multiplier double") \
    .write.mode("overwrite").saveAsTable(f"{FQ}.intervention_presets")

# Criteria are normalised to 0-100 (SPEC 8.2). P(collapse before judgment) scales the 24m probability to the expected duration.
spark.sql(f"""
CREATE OR REPLACE FUNCTION {FQ}.pps_compute(
  recoverable DOUBLE, litigation_cost DOUBLE, duration_months DOUBLE, p_success DOUBLE, p_collapse_12m DOUBLE, p_collapse_24m DOUBLE,
  consumers_affected DOUBLE, consumer_loss DOUBLE, vulnerable BOOLEAN, strategic_priority DOUBLE, months_to_limitation DOUBLE, recent_smr BOOLEAN,
  w_financial DOUBLE, w_success DOUBLE, w_harm DOUBLE, w_deterrence DOUBLE, w_urgency DOUBLE)
RETURNS STRUCT<total: DOUBLE, band: STRING, financial: DOUBLE, success: DOUBLE, harm: DOUBLE, deterrence: DOUBLE, urgency: DOUBLE,
               p_collapse_before_judgment: DOUBLE, expected_recovery: DOUBLE, expected_net_recovery: DOUBLE, return_ratio: DOUBLE>
COMMENT 'Prosecution Priority Score (SPEC 8.2). A decision aid only; it never makes the decision.'
RETURN (
  WITH c AS (SELECT
    1 - pow(1 - least(greatest(p_collapse_24m, 0), 0.999), duration_months / 24.0) AS pcb,
    recoverable * p_success * (1 - (1 - pow(1 - least(greatest(p_collapse_24m, 0), 0.999), duration_months / 24.0))) AS er)
  SELECT named_struct(
    'total', round(w_financial * fin + w_success * suc + w_harm * hrm + w_deterrence * det + w_urgency * urg, 1),
    'band', CASE WHEN w_financial * fin + w_success * suc + w_harm * hrm + w_deterrence * det + w_urgency * urg >= 70 THEN 'Proceed'
                 WHEN w_financial * fin + w_success * suc + w_harm * hrm + w_deterrence * det + w_urgency * urg >= 50 THEN 'Proceed - conditional'
                 WHEN w_financial * fin + w_success * suc + w_harm * hrm + w_deterrence * det + w_urgency * urg >= 35 THEN 'Consider alternatives'
                 ELSE 'Do not pursue' END,
    'financial', round(fin, 1), 'success', round(suc, 1), 'harm', round(hrm, 1), 'deterrence', round(det, 1), 'urgency', round(urg, 1),
    'p_collapse_before_judgment', round(pcb, 4), 'expected_recovery', round(er, 0), 'expected_net_recovery', round(er - litigation_cost, 0),
    'return_ratio', round(er / nullif(litigation_cost, 0), 3))
  FROM (SELECT pcb, er,
    100 * least(er / nullif(litigation_cost, 0) / 3.0, 1.0) AS fin,
    100 * p_success AS suc,
    100 * (0.4 * least(log10(consumers_affected + 1) / 5, 1) + 0.4 * least(log10(consumer_loss + 1) / 8, 1) + 0.2 * CAST(vulnerable AS INT)) AS hrm,
    least(greatest(strategic_priority, 0), 100) AS det,
    100 * (0.6 * p_collapse_12m + 0.2 * CAST(recent_smr AS INT) + 0.2 * least(greatest((60 - months_to_limitation) / 48.0, 0), 1)) AS urg
    FROM c)
)""")

In [0]:
from sklearn.linear_model import LogisticRegression
hm = spark.table(f"{FQ}.silver_historical_matters").filter("success IS NOT NULL AND evidence_strength IS NOT NULL").toPandas()
lr = LogisticRegression().fit(hm[["evidence_strength"]].astype(float), hm["success"].astype(int))
B0, B1 = float(lr.intercept_[0]), float(lr.coef_[0][0])
print(f"P(success) = sigmoid({B0:.3f} + {B1:.3f} * evidence_strength), n={len(hm)}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW case_base AS
SELECT c.*, w.*, r.p_collapse_12m, r.p_collapse_24m, r.model_version,
       coalesce(c.counsel_evidence_strength, c.evidence_element_coverage, 0.5) AS evidence_strength_used,
       1 / (1 + exp(-({B0} + {B1} * coalesce(c.counsel_evidence_strength, c.evidence_element_coverage, 0.5)))) AS p_success_base,
       months_between(c.limitation_date, current_date()) AS months_to_limitation
FROM {FQ}.gold_case_inputs c
LEFT JOIN {FQ}.risk_score r ON r.entity_id = c.respondent_entity_id
CROSS JOIN {FQ}.scoring_weights w WHERE w.weights_version = '{WEIGHTS_VERSION}'""")

spark.sql(f"""
CREATE OR REPLACE TABLE {FQ}.pps_score AS
SELECT case_id, weights_version, model_version, s.total, s.band, to_json(s) AS criteria_json,
       p_success_base AS p_success, p_collapse_12m, p_collapse_24m, current_timestamp() AS scored_at, 'derived' AS provenance
FROM (SELECT *, {FQ}.pps_compute(est_recoverable_aud, est_litigation_cost_aud, est_duration_months, p_success_base,
        coalesce(p_collapse_12m, 0.1), coalesce(p_collapse_24m, 0.2), consumers_affected, total_consumer_loss_aud, vulnerable_cohort,
        strategic_priority_score, months_to_limitation, coalesce(recent_smr_flag, false),
        w_financial, w_success, w_harm, w_deterrence, w_urgency) AS s FROM case_base)""")

# Baseline plus one scenario per intervention preset (stamped with inputs, author and versions so it can be reproduced)
spark.sql(f"""
CREATE OR REPLACE TABLE {FQ}.scenario AS
WITH inp AS (
  SELECT b.case_id, 'Baseline' AS name, NULL AS preset_id, est_recoverable_aud AS rec, est_litigation_cost_aud AS cost,
         est_duration_months AS dur, p_success_base AS ps, coalesce(p_collapse_12m, 0.1) AS p12, coalesce(p_collapse_24m, 0.2) AS p24, b.*
  FROM case_base b
  UNION ALL
  SELECT b.case_id, p.name, p.preset_id, est_recoverable_aud * p.recoverable_multiplier, est_litigation_cost_aud * p.cost_multiplier,
         est_duration_months * p.duration_multiplier, coalesce(p.p_success_override, p_success_base),
         coalesce(p.p_collapse_override, p_collapse_12m, 0.1), coalesce(p.p_collapse_override, p_collapse_24m, 0.2), b.*
  FROM case_base b CROSS JOIN {FQ}.intervention_presets p)
SELECT uuid() AS scenario_id, case_id, name, preset_id, 'system' AS author,
       to_json(named_struct('recoverable', rec, 'litigation_cost', cost, 'duration_months', dur, 'p_success', ps, 'p_collapse_12m', p12, 'p_collapse_24m', p24)) AS inputs_json,
       to_json({FQ}.pps_compute(rec, cost, dur, ps, p12, p24, consumers_affected, total_consumer_loss_aud, vulnerable_cohort, strategic_priority_score,
               months_to_limitation, coalesce(recent_smr_flag, false), w_financial, w_success, w_harm, w_deterrence, w_urgency)) AS outputs_json,
       model_version, weights_version, current_timestamp() AS created_at
FROM inp""")
display(spark.sql(f"SELECT band, count(*) n, round(avg(total),1) avg_pps FROM {FQ}.pps_score GROUP BY band"))